# RAG Hands-On (Ungraded Practice): Document Preprocessing & Chunking Strategies

This is a short, **ungraded** practice notebook that follows up on the main RAG hands-on session.
It focuses on the two building blocks that everything else in a RAG pipeline depends on:

1. **Document preprocessing** - turning messy raw files/text into clean, uniform `Document` objects.
2. **Chunking strategies** — splitting long documents into retrievable pieces, and comparing the trade-offs.

There is no dataset download and no vector index in this notebook.

Cells marked **`# TODO`** are for you to fill in. Everything else runs
as-is so you always have a working baseline to build on.

**How to use this notebook**
- Run cells top to bottom.
- Wherever you see `# TODO`, replace the `...` or `pass` with your own implementation.
- Each exercise has a short "sanity check" cell right after it. If it prints something obviously
  wrong (errors, empty lists, etc.) revisit your code.
- There are a few short reflection questions at the end. Answer them in the markdown cell provided.


## 0 · Setup

Only lightweight dependencies are needed for this notebook (no FAISS, no embeddings, no dataset
download).

In [ ]:
%pip install -q "langchain-text-splitters>=0.3,<0.4" "langchain-core>=0.3,<0.4"
print("Setup complete.")

In [ ]:
import re
import textwrap
from dataclasses import dataclass, field
from typing import List, Callable

from langchain_core.documents import Document

print("Imports ready.")

## 1 · Sample "raw" documents

To keep this notebook self-contained, we use a few short synthetic documents that stand in for
messy real-world text: inconsistent whitespace, stray HTML-ish tags, page-break artifacts, and
mixed casing are the kind of noises a real ingestion pipeline has to clean up before chunking.

Don't edit this cell. just run it.

In [ ]:
RAW_DOCS = {
    "rag_intro": """
        Retrieval-Augmented   Generation (RAG)    grounds a language model in an
        external corpus.\n\n

        Instead   of relying only on parametric memory, the system retrieves relevant
        passages at   query   time and conditions its answer on them.<br>

        This   reduces hallucination and lets the system cite sources.\n\n\n
        Page 1 of 3
    """,
    "chunking_notes": """
        CHUNKING   STRATEGIES\n
        A chunk is the atomic unit that gets embedded and retrieved.<p>
        Fixed-size chunking splits text into equal-sized windows, optionally with
        overlap.  \n\n Sliding-window chunking groups a fixed number of sentences and
        slides   forward   by a smaller step, so consecutive chunks share context.\n\n
        Semantic     chunking instead splits where the *meaning* shifts between
        consecutive sentences, so chunk boundaries follow topic changes rather than a
        fixed budget.\n\n\n Page 2 of 3
    """,
    "eval_notes": """
        EVALUATING   RETRIEVAL\n\n
        A retrieved chunk counts as relevant to a question when it comes from the
        right document AND shares a substantial fraction of its words with the gold
        evidence passage.   <br> Hit@k and MRR are the two most common metrics used
        to compare chunking   strategies.\n\n\n Page 3 of 3
    """,
}

for name, text in RAW_DOCS.items():
    print(f"--- {name} (raw, {len(text)} chars) ---")
    print(repr(text[:120]), "...\n")

## 2 · Document preprocessing

Before any chunking or embedding happens, raw text needs to be **cleaned and normalized**. Typical
steps include:

- collapsing repeated whitespace/newlines into single spaces (or paragraph breaks),
- stripping leftover markup artifacts (e.g. `<br>`, `<p>`),
- removing boilerplate like page numbers/footers (`"Page 1 of 3"`),
- normalizing whitespace at the start/end of the text.

You will **not** lowercase or remove punctuation here — that kind of normalization belongs to
tokenization/search, not to the clean text that gets embedded (we want to preserve the original
readable text for chunk display and for LLM context).

### Exercise 2.1 — Implement `clean_text`

Fill in `clean_text(text)` so that it:
1. Removes simple HTML-ish tags like `<br>`, `<p>`, `</p>` (a simple regex is enough — no need for
   a full HTML parser here).
2. Removes footer lines that look like `"Page X of Y"`.
3. Collapses any run of whitespace (spaces, tabs, newlines) into a single space.
4. Strips leading/trailing whitespace from the result.

In [ ]:
def clean_text(text: str) -> str:
    """Clean a raw document string.

    Steps:
      1. Strip simple HTML-ish tags (e.g. <br>, <p>, </p>).
      2. Remove 'Page X of Y' footer lines.
      3. Collapse all whitespace runs into a single space.
      4. Strip leading/trailing whitespace.
    """
    # TODO: implement the four steps described above.
    cleaned = text
    ...
    return cleaned

In [ ]:
# Sanity check — run this after implementing clean_text above.
for name, raw in RAW_DOCS.items():
    cleaned = clean_text(raw)
    print(f"--- {name} ---")
    print(textwrap.fill(cleaned, 100))
    print()
    assert "<br>" not in cleaned and "<p>" not in cleaned, "HTML-ish tags should be removed"
    assert "Page" not in cleaned, "Page footers should be removed"
    assert "  " not in cleaned, "Whitespace should be collapsed to single spaces"
print("clean_text looks good!")

### Exercise 2.2 — Build `Document` objects

Wrap each cleaned string into a LangChain `Document`, attaching useful metadata (at minimum a
`source` field with the dict key, e.g. `"rag_intro"`).

In [ ]:
def build_documents(raw_docs: dict) -> List[Document]:
    """Turn {name: raw_text} into a list of cleaned Document objects with metadata."""
    docs: List[Document] = []
    # TODO: for each (name, raw_text) pair — clean the text, then append a
    #       Document(page_content=..., metadata={"source": name}) to `docs`.
    ...
    return docs

In [ ]:
CORPUS = build_documents(RAW_DOCS)

# Sanity check
assert len(CORPUS) == len(RAW_DOCS), "Should produce one Document per raw doc"
for d in CORPUS:
    assert isinstance(d, Document)
    assert "source" in d.metadata
    print(d.metadata["source"], "->", len(d.page_content), "chars")
print("\nbuild_documents looks good!")

## 3 · Chunking strategies

Now that `CORPUS` holds clean `Document`s, split each one into chunks. We'll implement and compare
three strategies from the main session, on this small corpus:

- **Fixed-size chunking** — hard character windows with overlap.
- **Sliding-window (sentence-based) chunking** — group N sentences, slide forward by a smaller step.
- **Recursive chunking** — LangChain's `RecursiveCharacterTextSplitter`, which tries paragraph →
  sentence → word boundaries before falling back to hard cuts (already implemented for you, as a
  reference/comparison point).

### Exercise 3.1 — Fixed-size chunking

Implement `fixed_chunks(text, size, overlap)` that returns a list of substrings, each up to `size`
characters long, where consecutive chunks overlap by `overlap` characters.

Example: `fixed_chunks("ABCDEFGHIJ", size=4, overlap=1)` should return
`["ABCD", "DEFG", "GHIJ"]`.

In [ ]:
def fixed_chunks(text: str, size: int, overlap: int) -> List[str]:
    """Split `text` into fixed-size windows of `size` chars with `overlap` chars shared
    between consecutive windows."""
    assert 0 <= overlap < size, "overlap must be smaller than size"
    chunks: List[str] = []
    # TODO: slide a window of length `size` across `text`, stepping by (size - overlap)
    #       each time, and append each window's text to `chunks`.
    ...
    return chunks

In [ ]:
# Sanity check
example = fixed_chunks("ABCDEFGHIJ", size=4, overlap=1)
print("fixed_chunks('ABCDEFGHIJ', size=4, overlap=1) ->", example)
assert example == ["ABCD", "DEFG", "GHIJ"], f"Got {example}"
print("fixed_chunks looks good!")

### Exercise 3.2 — Sliding-window (sentence-based) chunking

Implement `sliding_chunks(text, n_sentences, step)` that:
1. Splits `text` into sentences (a simple split on `.`, `!`, `?` followed by whitespace is enough).
2. Groups `n_sentences` consecutive sentences into a chunk.
3. Slides forward by `step` sentences (`step < n_sentences`, so chunks overlap).

Example, with sentences `[S1, S2, S3, S4, S5]`, `n_sentences=3`, `step=2`:
chunk 1 = `S1 S2 S3`, chunk 2 = `S3 S4 S5`.

In [ ]:
_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

def split_sentences(text: str) -> List[str]:
    """Very simple sentence splitter (good enough for this exercise)."""
    return [s.strip() for s in _SENT_SPLIT_RE.split(text) if s.strip()]

def sliding_chunks(text: str, n_sentences: int, step: int) -> List[str]:
    """Group sentences into overlapping windows of `n_sentences`, advancing by `step`."""
    assert 0 < step < n_sentences, "step must be smaller than n_sentences"
    sentences = split_sentences(text)
    chunks: List[str] = []
    # TODO: slide a window of `n_sentences` sentences across `sentences`, stepping by
    #       `step`, join each window's sentences with a space, and append to `chunks`.
    ...
    return chunks

In [ ]:
# Sanity check
demo_sentences = "S1. S2. S3. S4. S5."
example = sliding_chunks(demo_sentences, n_sentences=3, step=2)
print("sliding_chunks(...) ->", example)
assert example[0] == "S1. S2. S3.", f"Got {example}"
assert example[1] == "S3. S4. S5.", f"Got {example}"
print("sliding_chunks looks good!")

### 3.3 — Recursive chunking (reference implementation)

This one is done for you, as a reference: LangChain's `RecursiveCharacterTextSplitter` tries to
split on paragraph breaks first, then sentences, then words, only falling back to a hard character
cut if nothing else fits within `chunk_size`.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def recursive_chunks(text: str, size: int, overlap: int) -> List[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_text(text)

# Quick look
sample = CORPUS[0].page_content
print(recursive_chunks(sample, size=80, overlap=20))

## 4 · Compare the strategies

### Exercise 4.1 — Run all three strategies on the same document

Using `CORPUS[1]` (the `chunking_notes` document), produce chunks with each of the three
strategies below, using comparable settings, and print how many chunks each produces and their
average length.

In [ ]:
sample_doc = CORPUS[1].page_content
print("SAMPLE DOCUMENT:\n" + textwrap.fill(sample_doc, 100) + "\n")

strategies = {
    "fixed (120/20)":     lambda t: fixed_chunks(t, size=120, overlap=20),
    "sliding (2 sent/1)": lambda t: sliding_chunks(t, n_sentences=2, step=1),
    "recursive (120/20)": lambda t: recursive_chunks(t, size=120, overlap=20),
}

# TODO: for each (name, fn) pair in `strategies`, call fn(sample_doc) and print:
#   - the strategy name
#   - the number of chunks produced
#   - the average chunk length (in characters)
#   - each chunk, truncated to ~80 characters for readability
...


### Exercise 4.2 — Chunk the whole corpus

Write `chunk_corpus(docs, splitter_fn)` that applies a `text -> List[str]` splitter function to
every `Document` in `docs`, and returns a flat list of new `Document` chunk objects. Each chunk's
metadata should include:
- `source`: copied from the parent document,
- `chunk_id`: the index of the chunk within its parent document (starting at 0).

Drop any chunk shorter than 15 characters (likely a sliver with no useful content).

In [ ]:
def chunk_corpus(docs: List[Document], splitter_fn: Callable[[str], List[str]]) -> List[Document]:
    """Apply `splitter_fn` to every doc in `docs`, returning a flat list of chunk Documents."""
    chunks: List[Document] = []
    # TODO: for each parent Document, split its page_content with `splitter_fn`, and for
    #       every non-trivial piece (>= 15 chars after stripping) append a new Document
    #       with metadata {"source": ..., "chunk_id": ...}.
    ...
    return chunks

In [ ]:
fixed_chunk_docs = chunk_corpus(CORPUS, lambda t: fixed_chunks(t, size=150, overlap=30))
recursive_chunk_docs = chunk_corpus(CORPUS, lambda t: recursive_chunks(t, size=150, overlap=30))

print(f"fixed chunking     -> {len(fixed_chunk_docs)} chunks total")
print(f"recursive chunking -> {len(recursive_chunk_docs)} chunks total")

# Sanity check
for d in fixed_chunk_docs[:3]:
    assert "source" in d.metadata and "chunk_id" in d.metadata
    print(d.metadata, "->", repr(d.page_content[:60]))
print("\nchunk_corpus looks good!")

## 5 · Reflection (short answers)

Answer briefly in the cell below (2-3 sentences each is plenty). This part is not graded — it's a
prompt to consolidate what you just built.

1. Looking at the fixed-size chunks you printed in Exercise 4.1, do any of them cut off mid-word or
   mid-sentence? Why does that happen, and why might it hurt retrieval quality?
2. Sliding-window chunking and recursive chunking both preserve more "natural" boundaries than
   fixed-size chunking, but for different reasons. What's the difference between the two
   approaches?
3. If you were chunking a corpus of legal contracts (long, precise, clause-based) versus a corpus
   of chat transcripts (short, conversational turns), would you use the same chunk size and
   strategy for both? What would you change and why?

**Your answers:**

1. _..._
2. _..._
3. _..._